## PART 1: SETUP & DOWNLOAD DATA USING GDOWN

In [1]:
print("🔧 Installing required libraries...")

# Install libraries
!pip install -q gdown pandas numpy scikit-learn matplotlib seaborn imbalanced-learn lightgbm

print("\n✓ Libraries installed!")

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gdown
import os
import gc
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported!")

🔧 Installing required libraries...

✓ Libraries installed!
✓ Libraries imported!


DOWNLOAD DATASETS USING GDOWN

In [2]:
print("📥 Downloading datasets from Google Drive...")
print("This may take several minutes depending on file size...\n")

TRAIN_FILE_ID = '1Ujh37a1kTarNf3dSOSrj-hKDswULebYA'
TEST_FILE_ID = '1ALveidCmKzk4p_liQ7b663BXL0PcTaFN'

# Output filenames
TRAIN_FILE = 'train_transaction.csv'
TEST_FILE = 'test_transaction.csv'

# Download train_transaction.csv
print("📥 Downloading train_transaction.csv...")
train_url = f'https://drive.google.com/uc?id={TRAIN_FILE_ID}'
gdown.download(train_url, TRAIN_FILE, quiet=False)
print(f"✓ Train file downloaded: {TRAIN_FILE}")

# Download test_transaction.csv
print("\n📥 Downloading test_transaction.csv...")
test_url = f'https://drive.google.com/uc?id={TEST_FILE_ID}'
gdown.download(test_url, TEST_FILE, quiet=False)
print(f"✓ Test file downloaded: {TEST_FILE}")

# Verify files exist
if os.path.exists(TRAIN_FILE) and os.path.exists(TEST_FILE):
    print("\n✅ All files downloaded successfully!")
    print(f"Train file size: {os.path.getsize(TRAIN_FILE) / 1024**2:.2f} MB")
    print(f"Test file size: {os.path.getsize(TEST_FILE) / 1024**2:.2f} MB")
else:
    print("\n❌ Error: Files not downloaded properly")

📥 Downloading datasets from Google Drive...
This may take several minutes depending on file size...

📥 Downloading train_transaction.csv...


Downloading...
From (original): https://drive.google.com/uc?id=1Ujh37a1kTarNf3dSOSrj-hKDswULebYA
From (redirected): https://drive.google.com/uc?id=1Ujh37a1kTarNf3dSOSrj-hKDswULebYA&confirm=t&uuid=af566f81-3e5d-4540-bd5e-690059dbe053
To: /content/train_transaction.csv
100%|██████████| 683M/683M [00:08<00:00, 78.9MB/s]


✓ Train file downloaded: train_transaction.csv

📥 Downloading test_transaction.csv...


Downloading...
From (original): https://drive.google.com/uc?id=1ALveidCmKzk4p_liQ7b663BXL0PcTaFN
From (redirected): https://drive.google.com/uc?id=1ALveidCmKzk4p_liQ7b663BXL0PcTaFN&confirm=t&uuid=e3143d59-b9ef-4d75-bece-1640e441a10a
To: /content/test_transaction.csv
100%|██████████| 613M/613M [00:07<00:00, 84.1MB/s]

✓ Test file downloaded: test_transaction.csv

✅ All files downloaded successfully!
Train file size: 651.69 MB
Test file size: 584.79 MB


LOAD DATASETS

In [3]:
print("\n📊 Loading datasets into memory...")

try:
    # Load train data
    print("Loading train_transaction.csv...")
    train = pd.read_csv(TRAIN_FILE)
    print(f"✓ Train loaded: {train.shape[0]:,} rows × {train.shape[1]} columns")

    # Load test data
    print("\nLoading test_transaction.csv...")
    test = pd.read_csv(TEST_FILE)
    print(f"✓ Test loaded: {test.shape[0]:,} rows × {test.shape[1]} columns")

    print("\n✅ Datasets loaded successfully!")

except Exception as e:
    print(f"\n❌ Error loading data: {e}")


📊 Loading datasets into memory...
Loading train_transaction.csv...
✓ Train loaded: 590,540 rows × 394 columns

Loading test_transaction.csv...
✓ Test loaded: 506,691 rows × 393 columns

✅ Datasets loaded successfully!


INITIAL EXPLORATION

In [4]:
print("📊 INITIAL DATA EXPLORATION")

# Basic info
print("\n1️⃣ TRAIN DATA OVERVIEW")
print("-" * 40)
print(f"Shape: {train.shape}")
print(f"Memory usage: {train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display columns
print("\n📝 Column names (first 20):")
print(train.columns.tolist()[:20])

# Target distribution
print("\n2️⃣ TARGET DISTRIBUTION")
print("-" * 40)
fraud_counts = train['isFraud'].value_counts()
print(fraud_counts)
print(f"\nFraud rate: {train['isFraud'].mean()*100:.4f}%")
print(f"Class imbalance ratio: 1:{fraud_counts[0]/fraud_counts[1]:.1f}")

# Column types
print("\n3️⃣ COLUMN TYPES")
print("-" * 40)
print(f"Numeric columns: {train.select_dtypes(include=[np.number]).shape[1]}")
print(f"Categorical columns: {train.select_dtypes(include=['object']).shape[1]}")

# Missing values
print("\n4️⃣ MISSING VALUES (Top 15)")
print("-" * 40)
missing = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_%': missing.values
}).head(15)
print(missing_df.to_string(index=False))

# Basic statistics for key columns
print("\n5️⃣ KEY STATISTICS")
print("-" * 40)
if 'TransactionAmt' in train.columns:
    print(f"TransactionAmt - Mean: ${train['TransactionAmt'].mean():.2f}, Median: ${train['TransactionAmt'].median():.2f}")
if 'TransactionDT' in train.columns:
    print(f"TransactionDT - Range: {train['TransactionDT'].min()} to {train['TransactionDT'].max()}")

# Sample data
print("\n6️⃣ SAMPLE DATA (First 3 rows)")
print("-" * 40)
print(train.head(3).to_string())

print("\n" + "="*40)
print("✓ Part 1 Complete!")
print("="*40)

📊 INITIAL DATA EXPLORATION

1️⃣ TRAIN DATA OVERVIEW
----------------------------------------
Shape: (590540, 394)
Memory usage: 2062.07 MB

📝 Column names (first 20):
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3']

2️⃣ TARGET DISTRIBUTION
----------------------------------------
isFraud
0    569877
1     20663
Name: count, dtype: int64

Fraud rate: 3.4990%
Class imbalance ratio: 1:27.6

3️⃣ COLUMN TYPES
----------------------------------------
Numeric columns: 380
Categorical columns: 14

4️⃣ MISSING VALUES (Top 15)
----------------------------------------
Column  Missing_%
 dist2  93.628374
    D7  93.409930
   D13  89.509263
   D14  89.469469
   D12  89.041047
    D6  87.606767
    D9  87.312290
    D8  87.312290
  V153  86.123717
  V149  86.123717
  V141  86.123717
  V146  86.123717
  V154  86.123717
  V162  86.12

## PART 2: DATA PREPROCESSING & CLEANING

In [5]:
print("🧹 Starting data preprocessing...")

# Create copies to preserve original
train_clean = train.copy()
test_clean = test.copy()

print(f"Original shapes - Train: {train_clean.shape}, Test: {test_clean.shape}")

🧹 Starting data preprocessing...
Original shapes - Train: (590540, 394), Test: (506691, 393)


STEP 1: Separate target and IDs

In [6]:
y_train = train_clean['isFraud'].copy()
train_ids = train_clean['TransactionID'].copy()
test_ids = test_clean['TransactionID'].copy()

# Drop from features
train_clean = train_clean.drop(['TransactionID', 'isFraud'], axis=1)
test_clean = test_clean.drop(['TransactionID'], axis=1)

print("\n✓ Target and IDs separated")
print(f"  - Features shape: {train_clean.shape}")
print(f"  - Target shape: {y_train.shape}")


✓ Target and IDs separated
  - Features shape: (590540, 392)
  - Target shape: (590540,)


STEP 2: Align columns between train and test

In [7]:
print("\n🔄 Aligning columns between train and test...")

# Get common columns
common_cols = list(set(train_clean.columns) & set(test_clean.columns))
train_clean = train_clean[common_cols]
test_clean = test_clean[common_cols]

print(f"✓ Aligned to {len(common_cols)} common columns")


🔄 Aligning columns between train and test...
✓ Aligned to 392 common columns


STEP 3: Remove high missing columns (>95%)

In [8]:
print("\n🗑️ Removing high missing columns...")

missing_pct = train_clean.isnull().sum() / len(train_clean)
high_missing_cols = missing_pct[missing_pct > 0.95].index.tolist()

if high_missing_cols:
    print(f"Removing {len(high_missing_cols)} columns with >95% missing:")
    print(f"  Examples: {high_missing_cols[:5]}")
    train_clean = train_clean.drop(columns=high_missing_cols)
    test_clean = test_clean.drop(columns=high_missing_cols)
else:
    print("✓ No columns with >95% missing")

print(f"Remaining columns: {len(train_clean.columns)}")


🗑️ Removing high missing columns...
✓ No columns with >95% missing
Remaining columns: 392


STEP 4: Identify column types

In [9]:
print("\n📊 Identifying column types...")

numeric_cols = train_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = train_clean.select_dtypes(include=['object']).columns.tolist()

print(f"  - Numeric: {len(numeric_cols)}")
print(f"  - Categorical: {len(categorical_cols)}")

if categorical_cols:
    print(f"  - Categorical examples: {categorical_cols[:5]}")



📊 Identifying column types...
  - Numeric: 378
  - Categorical: 14
  - Categorical examples: ['R_emaildomain', 'M6', 'M7', 'M9', 'card4']


STEP 5: Handle missing values

In [10]:
print("\n🔧 Handling missing values...")

# For numeric: fill with -999 (missing indicator)
print("  - Filling numeric columns with -999...")
for col in numeric_cols:
    if train_clean[col].isnull().sum() > 0:
        train_clean[col].fillna(-999, inplace=True)
        test_clean[col].fillna(-999, inplace=True)

# For categorical: fill with 'missing'
print("  - Filling categorical columns with 'missing'...")
for col in categorical_cols:
    if train_clean[col].isnull().sum() > 0:
        train_clean[col].fillna('missing', inplace=True)
        test_clean[col].fillna('missing', inplace=True)

print("✓ Missing values handled")

# Verify no missing values remain
remaining_missing = train_clean.isnull().sum().sum()
print(f"  - Remaining missing values: {remaining_missing}")

# Free memory
del train, test
gc.collect()

print("\n" + "="*40)
print("✓ Part 2 Complete!")
print("="*40)


🔧 Handling missing values...
  - Filling numeric columns with -999...
  - Filling categorical columns with 'missing'...
✓ Missing values handled
  - Remaining missing values: 0

✓ Part 2 Complete!


## PART 3: FEATURE ENGINEERING

In [11]:
print("\n🔨 Creating new features...\n")

for df in [train_clean, test_clean]:

    # 1. Transaction Amount features
    if 'TransactionAmt' in df.columns:
        print("  - Creating TransactionAmt features...")
        df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['TransactionAmt_rounded'] = df['TransactionAmt'].round(-2)  # Round to nearest 100

    # 2. Time-based features
    if 'TransactionDT' in df.columns:
        print("  - Creating time-based features...")
        df['Transaction_hour'] = (df['TransactionDT'] / 3600) % 24
        df['Transaction_day'] = (df['TransactionDT'] / (3600 * 24))
        df['Transaction_day_of_week'] = (df['Transaction_day'] % 7).astype(int)
        df['Transaction_is_weekend'] = df['Transaction_day_of_week'].isin([5, 6]).astype(int)

    # 3. Card features (if exist)
    if 'card1' in df.columns and 'card2' in df.columns:
        print("  - Creating card combination features...")
        df['card1_card2'] = df['card1'].astype(str) + '_' + df['card2'].astype(str)

    # 4. Address features
    if 'addr1' in df.columns and 'addr2' in df.columns:
        print("  - Creating address combination features...")
        df['addr1_addr2'] = df['addr1'].astype(str) + '_' + df['addr2'].astype(str)

    # 5. Email domain features (if exist)
    if 'P_emaildomain' in df.columns:
        print("  - Creating email domain features...")
        df['P_emaildomain_isNull'] = df['P_emaildomain'].isnull().astype(int)

    if 'R_emaildomain' in df.columns:
        df['R_emaildomain_isNull'] = df['R_emaildomain'].isnull().astype(int)

print("\n✓ Feature engineering complete!")
print(f"New feature count: {len(train_clean.columns)}")

# Update column types after feature engineering
numeric_cols = train_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = train_clean.select_dtypes(include=['object']).columns.tolist()

print(f"  - Numeric: {len(numeric_cols)}")
print(f"  - Categorical: {len(categorical_cols)}")

print("\n" + "="*40)
print("✓ Part 3 Complete!")
print("="*40)


🔨 Creating new features...

  - Creating TransactionAmt features...
  - Creating time-based features...
  - Creating card combination features...
  - Creating address combination features...
  - Creating email domain features...
  - Creating TransactionAmt features...
  - Creating time-based features...
  - Creating card combination features...
  - Creating address combination features...
  - Creating email domain features...

✓ Feature engineering complete!
New feature count: 403
  - Numeric: 387
  - Categorical: 16

✓ Part 3 Complete!


## PART 4: ENCODING CATEGORICAL VARIABLES

In [12]:
from sklearn.preprocessing import LabelEncoder

print("🔤 Encoding categorical variables...")

if len(categorical_cols) > 0:
    print(f"\nEncoding {len(categorical_cols)} categorical columns...")

    for i, col in enumerate(categorical_cols, 1):
        try:
            le = LabelEncoder()
            # Combine train and test for consistent encoding
            combined = pd.concat([train_clean[col], test_clean[col]], axis=0)
            le.fit(combined.astype(str))

            train_clean[col] = le.transform(train_clean[col].astype(str))
            test_clean[col] = le.transform(test_clean[col].astype(str))

            if i % 10 == 0:
                print(f"  - Encoded {i}/{len(categorical_cols)} columns...")
        except Exception as e:
            print(f"  ⚠️ Warning: Could not encode {col}: {e}")

    print(f"✓ All categorical columns encoded!")
else:
    print("✓ No categorical columns to encode")

# Verify all columns are numeric now
print(f"\nFinal data types:")
print(f"  - Numeric columns: {len(train_clean.select_dtypes(include=[np.number]).columns)}")
print(f"  - Non-numeric columns: {len(train_clean.select_dtypes(exclude=[np.number]).columns)}")

print(f"\nFinal shapes:")
print(f"  - Train: {train_clean.shape}")
print(f"  - Test: {test_clean.shape}")

# Free memory
gc.collect()

print("\n" + "="*40)
print("✓ Part 4 Complete!")
print("="*40)

🔤 Encoding categorical variables...

Encoding 16 categorical columns...
  - Encoded 10/16 columns...
✓ All categorical columns encoded!

Final data types:
  - Numeric columns: 403
  - Non-numeric columns: 0

Final shapes:
  - Train: (590540, 403)
  - Test: (506691, 403)

✓ Part 4 Complete!


## PART 5: FEATURE SELECTION

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

print("\n📊 Calculating feature importance...")
print("Using Random Forest on a sample for speed...")

# Take a sample (to avoid memory issues)
sample_size = min(50000, len(train_clean))
sample_indices = np.random.choice(len(train_clean), sample_size, replace=False)

X_sample = train_clean.iloc[sample_indices]
y_sample = y_train.iloc[sample_indices]

print(f"Sample size: {len(X_sample):,} rows")

# Train a quick Random Forest
print("\n🌲 Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf.fit(X_sample, y_sample)
print("✓ Model trained!")

# Get feature importance
print("\n📈 Extracting feature importance...")
feature_importance = pd.DataFrame({
    'feature': train_clean.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🏆 Top 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))


📊 Calculating feature importance...
Using Random Forest on a sample for speed...
Sample size: 50,000 rows

🌲 Training Random Forest...
✓ Model trained!

📈 Extracting feature importance...

🏆 Top 20 Most Important Features:
feature  importance
   V201    0.045541
   V257    0.042167
   V242    0.030326
   V258    0.027590
     C1    0.022555
    V45    0.022074
   V189    0.015319
     C7    0.014247
     C4    0.014201
   V244    0.014149
    C14    0.013497
    V44    0.013332
   V274    0.012870
   V188    0.012620
   V246    0.012596
   V200    0.012575
    V86    0.011863
    C12    0.011697
    C13    0.010889
     C8    0.009779


Select Top Features

In [14]:
# Select top N features (adjust based on memory)
n_features_to_keep = min(150, len(train_clean.columns))

selected_features = feature_importance.head(n_features_to_keep)['feature'].tolist()

print(f"\n✓ Selected {len(selected_features)} features out of {len(train_clean.columns)}")

# Apply feature selection
X_train_selected = train_clean[selected_features].copy()
X_test_selected = test_clean[selected_features].copy()

print(f"\nNew shapes:")
print(f"  - Train: {X_train_selected.shape}")
print(f"  - Test: {X_test_selected.shape}")

# Free memory
del train_clean, test_clean, X_sample, y_sample, rf
gc.collect()

print("\n" + "="*40)
print("✓ Part 5 Complete!")
print("="*40)


✓ Selected 150 features out of 403

New shapes:
  - Train: (590540, 150)
  - Test: (506691, 150)

✓ Part 5 Complete!


## PART 6: TRAIN/VAL SPLIT & HANDLE CLASS IMBALANCE

In [16]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Split data
print("\n✂️ Splitting data into train and validation...")

X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_selected,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"Train set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

print(f"\nClass distribution in train:")
print(y_train_split.value_counts())
print(f"Fraud rate: {y_train_split.mean()*100:.4f}%")

# Apply SMOTE to training set only
print("\n🔄 Applying SMOTE to balance training set...")
print("This may take a few minutes...")

# Use SMOTE with sampling_strategy to avoid too much oversampling
smote = SMOTE(
    sampling_strategy=0.3,  # Balance to 30% fraud (instead of 50/50)
    random_state=42,
    k_neighbors=5,
    #n_jobs=-1
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train_split)

print(f"\n✓ SMOTE complete!")
print(f"Original train: {X_train.shape}")
print(f"Balanced train: {X_train_balanced.shape}")

print(f"\nNew class distribution:")
print(pd.Series(y_train_balanced).value_counts())
print(f"New fraud rate: {y_train_balanced.mean()*100:.2f}%")

# Free memory
del X_train_selected, X_train, y_train_split
gc.collect()

print("\n" + "="*40)
print("✓ Part 6 Complete!")
print("="*40)


✂️ Splitting data into train and validation...
Train set: (472432, 150)
Validation set: (118108, 150)

Class distribution in train:
isFraud
0    455902
1     16530
Name: count, dtype: int64
Fraud rate: 3.4989%

🔄 Applying SMOTE to balance training set...
This may take a few minutes...

✓ SMOTE complete!
Original train: (472432, 150)
Balanced train: (592672, 150)

New class distribution:
isFraud
0    455902
1    136770
Name: count, dtype: int64
New fraud rate: 23.08%

✓ Part 6 Complete!


## PART 7: MODEL TRAINING - LIGHTGBM

In [17]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Define model parameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'max_depth': 10,
    'min_child_samples': 20,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1
}

print("\n📝 Model parameters:")
for key, value in params.items():
    print(f"  - {key}: {value}")

# Create LightGBM datasets
print("\n📦 Creating LightGBM datasets...")

train_data = lgb.Dataset(X_train_balanced, label=y_train_balanced)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

print("✓ Datasets created!")


📝 Model parameters:
  - objective: binary
  - metric: auc
  - boosting_type: gbdt
  - num_leaves: 31
  - learning_rate: 0.05
  - feature_fraction: 0.8
  - bagging_fraction: 0.8
  - bagging_freq: 5
  - max_depth: 10
  - min_child_samples: 20
  - verbose: -1
  - random_state: 42
  - n_jobs: -1

📦 Creating LightGBM datasets...
✓ Datasets created!


In [18]:
# Train model
print("\n🚀 Training model...")
print("This will take several minutes...\n")

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

print(f"\n✓ Training complete!")
print(f"Best iteration: {model.best_iteration}")
print(f"Best score: {model.best_score['val']['auc']:.4f}")

# Free memory
del train_data, val_data
gc.collect()

print("\n" + "="*40)
print("✓ Part 7 Complete!")
print("="*40)


🚀 Training model...
This will take several minutes...

Training until validation scores don't improve for 50 rounds
[100]	train's auc: 0.98476	val's auc: 0.882
[200]	train's auc: 0.988833	val's auc: 0.904158
[300]	train's auc: 0.991009	val's auc: 0.918973
[400]	train's auc: 0.99239	val's auc: 0.927721
[500]	train's auc: 0.99357	val's auc: 0.934248
[600]	train's auc: 0.99434	val's auc: 0.938607
[700]	train's auc: 0.9951	val's auc: 0.942216
[800]	train's auc: 0.995731	val's auc: 0.945683
[900]	train's auc: 0.996288	val's auc: 0.948712
[1000]	train's auc: 0.996759	val's auc: 0.951145
Did not meet early stopping. Best iteration is:
[1000]	train's auc: 0.996759	val's auc: 0.951145

✓ Training complete!
Best iteration: 1000
Best score: 0.9511

✓ Part 7 Complete!


## PART 8: MODEL EVALUATION

In [19]:
from sklearn.metrics import (roc_auc_score, precision_recall_curve, auc,
                              classification_report, confusion_matrix)

# Make predictions
print("\n🔮 Generating predictions...")

y_train_pred = model.predict(X_train_balanced, num_iteration=model.best_iteration)
y_val_pred = model.predict(X_val, num_iteration=model.best_iteration)

print("✓ Predictions generated!")

# Calculate metrics
print("\n📈 PERFORMANCE METRICS")
print("="*60)

# ROC AUC
train_auc = roc_auc_score(y_train_balanced, y_train_pred)
val_auc = roc_auc_score(y_val, y_val_pred)

print(f"\n1️⃣ ROC AUC Score:")
print(f"  - Train: {train_auc:.4f}")
print(f"  - Validation: {val_auc:.4f}")

# Precision-Recall AUC
precision, recall, _ = precision_recall_curve(y_val, y_val_pred)
pr_auc = auc(recall, precision)

print(f"\n2️⃣ Precision-Recall AUC:")
print(f"  - Validation: {pr_auc:.4f}")

# Classification Report (using 0.5 threshold)
y_val_pred_binary = (y_val_pred > 0.5).astype(int)

print(f"\n3️⃣ Classification Report (threshold=0.5):")
print(classification_report(y_val, y_val_pred_binary,
                          target_names=['Not Fraud', 'Fraud']))

# Confusion Matrix
cm = confusion_matrix(y_val, y_val_pred_binary)
print(f"\n4️⃣ Confusion Matrix:")
print(f"                Predicted")
print(f"                No    Yes")
print(f"Actual No    {cm[0,0]:6d} {cm[0,1]:6d}")
print(f"Actual Yes   {cm[1,0]:6d} {cm[1,1]:6d}")

# Feature Importance
print(f"\n5️⃣ TOP 20 IMPORTANT FEATURES")
print("="*60)

importance_df = pd.DataFrame({
    'feature': X_train_balanced.columns,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print(importance_df.head(20).to_string(index=False))

# Prediction distribution
print(f"\n6️⃣ PREDICTION DISTRIBUTION")
print("="*60)

print(f"Validation predictions:")
print(f"  - Min: {y_val_pred.min():.4f}")
print(f"  - Max: {y_val_pred.max():.4f}")
print(f"  - Mean: {y_val_pred.mean():.4f}")
print(f"  - Median: {np.median(y_val_pred):.4f}")

print(f"\nPredictions > 0.5: {(y_val_pred > 0.5).sum():,} ({(y_val_pred > 0.5).mean()*100:.2f}%)")
print(f"Actual fraud: {y_val.sum():,} ({y_val.mean()*100:.2f}%)")

print("\n" + "="*40)
print("✓ Part 8 Complete!")
print("="*40)


🔮 Generating predictions...
✓ Predictions generated!

📈 PERFORMANCE METRICS

1️⃣ ROC AUC Score:
  - Train: 0.9968
  - Validation: 0.9511

2️⃣ Precision-Recall AUC:
  - Validation: 0.7601

3️⃣ Classification Report (threshold=0.5):
              precision    recall  f1-score   support

   Not Fraud       0.98      1.00      0.99    113975
       Fraud       0.93      0.52      0.67      4133

    accuracy                           0.98    118108
   macro avg       0.96      0.76      0.83    118108
weighted avg       0.98      0.98      0.98    118108


4️⃣ Confusion Matrix:
                Predicted
                No    Yes
Actual No    113816    159
Actual Yes     1990   2143

5️⃣ TOP 20 IMPORTANT FEATURES
               feature    importance
                   C12 516099.228770
                  V279 369316.166747
                  V294 331514.830523
                   C14 311286.316633
                    C8 214800.150941
                  V317 204507.446766
                   C11

## PART 9: GENERATE PREDICTIONS FOR TEST SET

In [20]:
print("\n🔮 Predicting on test data...")
print(f"Test shape: {X_test_selected.shape}")

test_predictions = model.predict(X_test_selected, num_iteration=model.best_iteration)

print("✓ Predictions complete!")

# Analyze predictions
print(f"\n📊 PREDICTION STATISTICS")
print("="*60)

print(f"Total predictions: {len(test_predictions):,}")
print(f"Min: {test_predictions.min():.6f}")
print(f"Max: {test_predictions.max():.6f}")
print(f"Mean: {test_predictions.mean():.6f}")
print(f"Median: {np.median(test_predictions):.6f}")
print(f"\nPredictions > 0.5: {(test_predictions > 0.5).sum():,} ({(test_predictions > 0.5).mean()*100:.2f}%)")
print(f"Predictions > 0.1: {(test_predictions > 0.1).sum():,} ({(test_predictions > 0.1).mean()*100:.2f}%)")


🔮 Predicting on test data...
Test shape: (506691, 150)
✓ Predictions complete!

📊 PREDICTION STATISTICS
Total predictions: 506,691
Min: 0.000044
Max: 0.999954
Mean: 0.036784
Median: 0.007590

Predictions > 0.5: 9,587 (1.89%)
Predictions > 0.1: 30,072 (5.93%)


In [21]:
# Create submission file
print(f"\n📝 Creating submission file...")

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud': test_predictions
})

# Save submission
submission.to_csv('submission.csv', index=False)

print("✓ Submission file created: submission.csv")
print(f"\nSubmission shape: {submission.shape}")
print(f"\nFirst 10 predictions:")
print(submission.head(10).to_string(index=False))

print(f"\nLast 10 predictions:")
print(submission.tail(10).to_string(index=False))

print("\n" + "="*40)
print("✅ ALL PARTS COMPLETE!")
print("="*40)
print("\n🎉 Your submission file is ready: submission.csv")
print("📥 You can download it from the Files panel on the left")


📝 Creating submission file...
✓ Submission file created: submission.csv

Submission shape: (506691, 2)

First 10 predictions:
 TransactionID  isFraud
       3663549 0.001300
       3663550 0.008290
       3663551 0.006103
       3663552 0.004207
       3663553 0.001742
       3663554 0.007171
       3663555 0.015857
       3663556 0.063936
       3663557 0.001293
       3663558 0.010589

Last 10 predictions:
 TransactionID  isFraud
       4170230 0.024635
       4170231 0.031810
       4170232 0.001137
       4170233 0.003312
       4170234 0.008720
       4170235 0.024654
       4170236 0.004918
       4170237 0.007898
       4170238 0.012477
       4170239 0.009742

✅ ALL PARTS COMPLETE!

🎉 Your submission file is ready: submission.csv
📥 You can download it from the Files panel on the left
